# DeepCpf1: 100 trials using TXT-input saved splits

Run `DeepCpf1_create_and_export_fixed_split_TXT_input.ipynb` first.

This notebook loads:

- `results/deepcpf1_fixed_split/saved_splits/train_split.csv`
- `results/deepcpf1_fixed_split/saved_splits/validation_split.csv`
- `results/deepcpf1_fixed_split/saved_splits/unseen_split.csv`

It does not recreate or reshuffle the split.

The notebook automatically runs:

- **DeepCpf1** when `chromatin_accessibility` exists in the saved split files;
- **Seq-deepCpf1** when the chromatin column is absent.

For every one of 100 Optuna trials, it records only:

- validation MSE;
- unseen Pearson correlation;
- unseen Spearman correlation.


In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error
from torch.utils.data import DataLoader, Dataset

OPTUNA_SEED = 42
MODEL_SEED = 42

N_TRIALS = 100
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

BASE_OUTPUT_DIR = Path(
    "results/deepcpf1_fixed_split"
)

SPLIT_DIR = (
    BASE_OUTPUT_DIR / "saved_splits"
)

RESULTS_DIR = (
    BASE_OUTPUT_DIR / "trial_results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_FILE = (
    SPLIT_DIR / "train_split.csv"
)

VALIDATION_FILE = (
    SPLIT_DIR / "validation_split.csv"
)

UNSEEN_FILE = (
    SPLIT_DIR / "unseen_split.csv"
)

for path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    UNSEEN_FILE,
]:
    assert path.exists(), (
        f"Missing file: {path.resolve()}\n"
        "Run DeepCpf1_create_and_export_fixed_split_TXT_input.ipynb first."
    )

DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)
print("Results directory:", RESULTS_DIR.resolve())


## Load the exact saved train, validation, and unseen subsets


In [ ]:
train_data = pd.read_csv(
    TRAIN_FILE
)

validation_data = pd.read_csv(
    VALIDATION_FILE
)

unseen_data = pd.read_csv(
    UNSEEN_FILE
)

required_columns = {
    "target_context_34nt",
    "activity",
}

for subset_name, subset_df in [
    ("train", train_data),
    ("validation", validation_data),
    ("unseen", unseen_data),
]:
    missing_columns = (
        required_columns
        - set(subset_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{subset_name} split is missing columns: "
            f"{sorted(missing_columns)}"
        )

USE_CHROMATIN = (
    "chromatin_accessibility"
    in train_data.columns
)

if USE_CHROMATIN:
    for subset_name, subset_df in [
        ("validation", validation_data),
        ("unseen", unseen_data),
    ]:
        if (
            "chromatin_accessibility"
            not in subset_df.columns
        ):
            raise ValueError(
                f"{subset_name} split does not contain "
                "chromatin_accessibility, while the training split does."
            )

MODEL_MODE = (
    "DeepCpf1"
    if USE_CHROMATIN
    else "Seq-deepCpf1"
)

print("Model mode:", MODEL_MODE)
print("Training samples:", len(train_data))
print("Validation samples:", len(validation_data))
print("Unseen samples:", len(unseen_data))

display(
    pd.DataFrame(
        {
            "subset": [
                "train",
                "validation",
                "unseen",
            ],
            "n_samples": [
                len(train_data),
                len(validation_data),
                len(unseen_data),
            ],
        }
    )
)


## Validate and one-hot encode the 34-nt sequences


In [ ]:
BASE_TO_INDEX = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3,
}


def one_hot_encode(
    sequence_series,
):
    encoded = np.zeros(
        (
            len(sequence_series),
            4,
            34,
        ),
        dtype=np.float32,
    )

    for sample_index, sequence in enumerate(
        sequence_series.astype(str)
    ):
        sequence = (
            sequence.strip().upper()
        )

        if len(sequence) != 34:
            raise ValueError(
                f"Row {sample_index}: expected 34 nt, "
                f"received {len(sequence)} nt."
            )

        if not set(sequence).issubset(
            BASE_TO_INDEX
        ):
            raise ValueError(
                f"Row {sample_index}: invalid sequence "
                f"{sequence!r}. Only A/C/G/T are allowed."
            )

        for position, base in enumerate(
            sequence
        ):
            encoded[
                sample_index,
                BASE_TO_INDEX[base],
                position,
            ] = 1.0

    return encoded


X_train = one_hot_encode(
    train_data[
        "target_context_34nt"
    ]
)

X_validation = one_hot_encode(
    validation_data[
        "target_context_34nt"
    ]
)

X_unseen = one_hot_encode(
    unseen_data[
        "target_context_34nt"
    ]
)

y_train = train_data[
    "activity"
].to_numpy(
    dtype=np.float32
)

y_validation = validation_data[
    "activity"
].to_numpy(
    dtype=np.float32
)

y_unseen = unseen_data[
    "activity"
].to_numpy(
    dtype=np.float32
)

if USE_CHROMATIN:
    ca_train = train_data[
        "chromatin_accessibility"
    ].to_numpy(
        dtype=np.float32
    )

    ca_validation = validation_data[
        "chromatin_accessibility"
    ].to_numpy(
        dtype=np.float32
    )

    ca_unseen = unseen_data[
        "chromatin_accessibility"
    ].to_numpy(
        dtype=np.float32
    )

    for subset_name, values in [
        ("train", ca_train),
        ("validation", ca_validation),
        ("unseen", ca_unseen),
    ]:
        unique_values = set(
            np.unique(values).tolist()
        )

        if not unique_values.issubset(
            {0.0, 1.0}
        ):
            raise ValueError(
                f"{subset_name} chromatin values must be binary 0/1. "
                f"Found: {sorted(unique_values)}"
            )

else:
    ca_train = np.zeros(
        len(train_data),
        dtype=np.float32,
    )

    ca_validation = np.zeros(
        len(validation_data),
        dtype=np.float32,
    )

    ca_unseen = np.zeros(
        len(unseen_data),
        dtype=np.float32,
    )

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_unseen:", X_unseen.shape)


## Dataset and DeepCpf1 architecture


In [ ]:
class DeepCpf1Dataset(Dataset):
    def __init__(
        self,
        sequence_features,
        chromatin_features,
        labels,
    ):
        self.sequence_features = torch.tensor(
            sequence_features,
            dtype=torch.float32,
        )

        self.chromatin_features = torch.tensor(
            chromatin_features,
            dtype=torch.float32,
        ).view(-1, 1)

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return (
            self.sequence_features[index],
            self.chromatin_features[index],
            self.labels[index],
        )


class DeepCpf1Model(nn.Module):
    def __init__(
        self,
        conv_filters=80,
        kernel_size=5,
        dense_1=80,
        dense_2=40,
        dense_3=40,
        dropout=0.30,
        use_chromatin=True,
        chromatin_scale=100.0,
    ):
        super().__init__()

        self.use_chromatin = (
            use_chromatin
        )

        self.chromatin_scale = (
            chromatin_scale
        )

        self.conv = nn.Conv1d(
            in_channels=4,
            out_channels=conv_filters,
            kernel_size=kernel_size,
        )

        self.pool = nn.AvgPool1d(
            kernel_size=2,
            stride=2,
        )

        with torch.no_grad():
            dummy_input = torch.zeros(
                1,
                4,
                34,
            )

            flattened_size = (
                self.pool(
                    F.relu(
                        self.conv(
                            dummy_input
                        )
                    )
                )
                .flatten(start_dim=1)
                .shape[1]
            )

        self.sequence_fc1 = nn.Linear(
            flattened_size,
            dense_1,
        )

        self.sequence_fc2 = nn.Linear(
            dense_1,
            dense_2,
        )

        self.sequence_fc3 = nn.Linear(
            dense_2,
            dense_3,
        )

        if self.use_chromatin:
            self.chromatin_fc = nn.Linear(
                1,
                dense_3,
            )

        self.output_layer = nn.Linear(
            dense_3,
            1,
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        sequence_input,
        chromatin_input,
    ):
        sequence_embedding = F.relu(
            self.conv(
                sequence_input
            )
        )

        sequence_embedding = self.pool(
            sequence_embedding
        )

        sequence_embedding = (
            sequence_embedding
            .flatten(start_dim=1)
        )

        sequence_embedding = self.dropout(
            sequence_embedding
        )

        sequence_embedding = self.dropout(
            F.relu(
                self.sequence_fc1(
                    sequence_embedding
                )
            )
        )

        sequence_embedding = self.dropout(
            F.relu(
                self.sequence_fc2(
                    sequence_embedding
                )
            )
        )

        sequence_embedding = F.relu(
            self.sequence_fc3(
                sequence_embedding
            )
        )

        if self.use_chromatin:
            chromatin_embedding = F.relu(
                self.chromatin_fc(
                    chromatin_input
                    * self.chromatin_scale
                )
            )

            merged_embedding = (
                sequence_embedding
                * chromatin_embedding
            )

        else:
            merged_embedding = (
                sequence_embedding
            )

        merged_embedding = self.dropout(
            merged_embedding
        )

        return self.output_layer(
            merged_embedding
        ).view(-1)


## Reproducibility and metric helpers


In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


def safe_pearson(
    y_true,
    y_pred,
):
    if (
        len(y_true) < 2
        or np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        return float(
            pearsonr(
                y_true,
                y_pred,
            )[0]
        )


def safe_spearman(
    y_true,
    y_pred,
):
    if (
        len(y_true) < 2
        or np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        return float(
            spearmanr(
                y_true,
                y_pred,
            )[0]
        )


def predict_model(
    model,
    data_loader,
):
    model.eval()

    labels = []
    predictions = []

    with torch.no_grad():
        for (
            sequence_batch,
            chromatin_batch,
            label_batch,
        ) in data_loader:
            sequence_batch = (
                sequence_batch.to(
                    DEVICE
                )
            )

            chromatin_batch = (
                chromatin_batch.to(
                    DEVICE
                )
            )

            prediction_batch = model(
                sequence_batch,
                chromatin_batch,
            )

            labels.extend(
                label_batch.numpy()
            )

            predictions.extend(
                prediction_batch
                .detach()
                .cpu()
                .numpy()
            )

    return (
        np.asarray(
            labels,
            dtype=float,
        ),
        np.asarray(
            predictions,
            dtype=float,
        ),
    )


In [ ]:
train_dataset = DeepCpf1Dataset(
    X_train,
    ca_train,
    y_train,
)

validation_dataset = DeepCpf1Dataset(
    X_validation,
    ca_validation,
    y_validation,
)

unseen_dataset = DeepCpf1Dataset(
    X_unseen,
    ca_unseen,
    y_unseen,
)

print(
    "Datasets:",
    len(train_dataset),
    len(validation_dataset),
    len(unseen_dataset),
)


## Run 100 fixed-split Optuna trials

The split remains unchanged across all trials. Model and data-loader seeds are also fixed at 42, so differences among trials come from the sampled hyperparameters.


In [ ]:
trial_records = []
trial_model_states = {}


def objective(trial):
    set_all_seeds(
        MODEL_SEED
    )

    conv_filters = trial.suggest_int(
        "conv_filters",
        48,
        128,
        step=8,
    )

    kernel_size = trial.suggest_categorical(
        "kernel_size",
        [3, 5, 7],
    )

    dense_1 = trial.suggest_int(
        "dense_1",
        48,
        160,
        step=16,
    )

    dense_2 = trial.suggest_int(
        "dense_2",
        24,
        96,
        step=8,
    )

    dense_3 = trial.suggest_int(
        "dense_3",
        24,
        96,
        step=8,
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.5,
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        5e-3,
        log=True,
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64],
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-8,
        1e-3,
        log=True,
    )

    train_generator = torch.Generator()
    train_generator.manual_seed(
        MODEL_SEED
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=train_generator,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    unseen_loader = DataLoader(
        unseen_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    model = DeepCpf1Model(
        conv_filters=conv_filters,
        kernel_size=kernel_size,
        dense_1=dense_1,
        dense_2=dense_2,
        dense_3=dense_3,
        dropout=dropout,
        use_chromatin=USE_CHROMATIN,
        chromatin_scale=100.0,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    loss_function = nn.MSELoss()

    best_validation_mse = np.inf
    best_model_state = None
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        model.train()

        for (
            sequence_batch,
            chromatin_batch,
            label_batch,
        ) in train_loader:
            sequence_batch = (
                sequence_batch.to(
                    DEVICE
                )
            )

            chromatin_batch = (
                chromatin_batch.to(
                    DEVICE
                )
            )

            label_batch = (
                label_batch.to(
                    DEVICE
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            predictions = model(
                sequence_batch,
                chromatin_batch,
            )

            loss = loss_function(
                predictions,
                label_batch,
            )

            loss.backward()
            optimizer.step()

        (
            validation_labels,
            validation_predictions,
        ) = predict_model(
            model,
            validation_loader,
        )

        validation_mse = float(
            mean_squared_error(
                validation_labels,
                validation_predictions,
            )
        )

        trial.report(
            validation_mse,
            epoch,
        )

        if (
            validation_mse
            < best_validation_mse
        ):
            best_validation_mse = (
                validation_mse
            )

            best_epoch = epoch

            epochs_without_improvement = 0

            best_model_state = {
                key: value
                .detach()
                .cpu()
                .clone()
                for key, value
                in model.state_dict().items()
            }

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            break

    if best_model_state is None:
        raise RuntimeError(
            "No valid best model state was recorded."
        )

    model.load_state_dict(
        best_model_state
    )

    model = model.to(
        DEVICE
    )

    (
        unseen_labels,
        unseen_predictions,
    ) = predict_model(
        model,
        unseen_loader,
    )

    unseen_pearson = safe_pearson(
        unseen_labels,
        unseen_predictions,
    )

    unseen_spearman = safe_spearman(
        unseen_labels,
        unseen_predictions,
    )

    trial.set_user_attr(
        "best_epoch",
        int(best_epoch),
    )

    trial_records.append(
        {
            "trial": trial.number,
            "validation_mse": (
                best_validation_mse
            ),
            "unseen_pearson": (
                unseen_pearson
            ),
            "unseen_spearman": (
                unseen_spearman
            ),
        }
    )

    trial_model_states[
        trial.number
    ] = best_model_state

    print(
        f"Trial {trial.number:3d} | "
        f"Validation MSE: "
        f"{best_validation_mse:.6f} | "
        f"Unseen Pearson: "
        f"{unseen_pearson:.4f} | "
        f"Unseen Spearman: "
        f"{unseen_spearman:.4f} | "
        f"Best epoch: {best_epoch}"
    )

    return best_validation_mse


study = optuna.create_study(
    direction="minimize",
    sampler=(
        optuna.samplers.TPESampler(
            seed=OPTUNA_SEED
        )
    ),
    study_name=(
        f"{MODEL_MODE}_TXT_input_"
        "fixed_split_100_trials"
    ),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
)


## Save the three requested metrics and the best model


In [ ]:
results_df = (
    pd.DataFrame(
        trial_records
    )
    .sort_values(
        "trial"
    )
    .reset_index(
        drop=True
    )
)

results_df.to_csv(
    RESULTS_DIR
    / "all_100_trial_metrics.csv",
    index=False,
)

results_df[
    "validation_mse"
].to_csv(
    RESULTS_DIR
    / "Validation_loss.txt",
    index=False,
    header=False,
)

results_df[
    "unseen_pearson"
].to_csv(
    RESULTS_DIR
    / "Unseen_Pearson.txt",
    index=False,
    header=False,
)

results_df[
    "unseen_spearman"
].to_csv(
    RESULTS_DIR
    / "Unseen_Spearman.txt",
    index=False,
    header=False,
)

best_trial_number = (
    study.best_trial.number
)

best_parameters = (
    study.best_trial.params
)

best_model = DeepCpf1Model(
    conv_filters=(
        best_parameters[
            "conv_filters"
        ]
    ),
    kernel_size=(
        best_parameters[
            "kernel_size"
        ]
    ),
    dense_1=(
        best_parameters[
            "dense_1"
        ]
    ),
    dense_2=(
        best_parameters[
            "dense_2"
        ]
    ),
    dense_3=(
        best_parameters[
            "dense_3"
        ]
    ),
    dropout=(
        best_parameters[
            "dropout"
        ]
    ),
    use_chromatin=(
        USE_CHROMATIN
    ),
    chromatin_scale=100.0,
)

best_model.load_state_dict(
    trial_model_states[
        best_trial_number
    ]
)

best_row = results_df.loc[
    results_df["trial"]
    == best_trial_number
].iloc[0]

torch.save(
    {
        "model_mode": MODEL_MODE,
        "use_chromatin": (
            USE_CHROMATIN
        ),
        "model_state_dict": (
            best_model.state_dict()
        ),
        "best_trial": int(
            best_trial_number
        ),
        "best_parameters": (
            best_parameters
        ),
        "best_epoch": int(
            study.best_trial
            .user_attrs[
                "best_epoch"
            ]
        ),
        "validation_mse": float(
            best_row[
                "validation_mse"
            ]
        ),
        "unseen_pearson": float(
            best_row[
                "unseen_pearson"
            ]
        ),
        "unseen_spearman": float(
            best_row[
                "unseen_spearman"
            ]
        ),
    },
    RESULTS_DIR
    / "best_DeepCpf1_model.pt",
)

with open(
    RESULTS_DIR
    / "best_trial_summary.json",
    "w",
) as handle:
    json.dump(
        {
            "model_mode": MODEL_MODE,
            "best_trial": int(
                best_trial_number
            ),
            "best_parameters": (
                best_parameters
            ),
            "best_epoch": int(
                study.best_trial
                .user_attrs[
                    "best_epoch"
                ]
            ),
            "validation_mse": float(
                best_row[
                    "validation_mse"
                ]
            ),
            "unseen_pearson": float(
                best_row[
                    "unseen_pearson"
                ]
            ),
            "unseen_spearman": float(
                best_row[
                    "unseen_spearman"
                ]
            ),
        },
        handle,
        indent=2,
    )

display(
    results_df.head()
)

print(
    "\nModel mode:",
    MODEL_MODE,
)

print(
    "Best trial:",
    best_trial_number,
)

print(
    "Best parameters:",
    best_parameters,
)

print(
    f"Validation MSE: "
    f"{best_row['validation_mse']:.6f}"
)

print(
    f"Unseen Pearson: "
    f"{best_row['unseen_pearson']:.4f}"
)

print(
    f"Unseen Spearman: "
    f"{best_row['unseen_spearman']:.4f}"
)

print(
    "\nSaved to:",
    RESULTS_DIR.resolve(),
)
